# M0 — 1.000 bộ 30 tuyến ngẫu nhiên

Đối chứng với bộ 30 cố định, cùng pool 60 và quota A10/B8/C12. Chọn kernel **Python thường** có NumPy và Matplotlib. Không cần khởi động PyQGIS cho bước này.


In [ ]:
from pathlib import Path
import sys

WORK = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / 'scripts/lez/selection_random.py').is_file()), None)
if WORK is None:
    raise RuntimeError('Mở notebook từ thư mục work hoặc work/notebooks.')
sys.path.insert(0, str(WORK / 'scripts'))
from lez.selection_random import preflight_random, run_random
from lez.common import input_path

print('Thư mục dự án:', WORK)


### 1. Kiểm tra đầu vào

Xác nhận hash của dữ liệu cố định và các gate GIS đã lưu. Cấu hình số bộ, seed và giới hạn lần bốc nằm trong `config/selection_random.json`.


In [ ]:
check = preflight_random(WORK)
for key in ('status', 'runtime', 'network_calls', 'qgis_started'):
    print(key, ':', check[key])
print({k: check['protocol'][k] for k in ('valid_samples', 'random_seed', 'max_attempts')})


### 2. Bốc đủ bộ hợp lệ và tính chỉ tiêu

Bốc đều theo từng nhóm quota, loại cả bộ nếu vi phạm ngưỡng trùng hành lang. Không lọc theo D, số ô hay độ đa dạng. Chạy lại để tiếp tục checkpoint hoặc dùng cache hợp lệ; bộ 30 gốc được giữ nguyên.


In [ ]:
report = run_random(WORK)
print(report['status'])
print('Bộ hợp lệ:', report['valid_samples'], '| Số lần bốc:', report['attempts'])
print('Bị loại:', report['rejections'], '| Seed:', report['random_seed'])


### 3. So sánh với bộ 30 cố định

D càng nhỏ thì cơ cấu chiều dài RC càng gần mạng tham chiếu. Tỷ lệ bộ ngẫu nhiên có D không lớn hơn bộ gốc là thống kê mô tả; không diễn giải như p-value. Quan sát 0/1.000 không có nghĩa xác suất bằng 0.


In [ ]:
c = report['comparison']
d = c['D_distribution']
print(f"D bộ gốc: {c['baseline_D']:.6f}")
print(f"M0 — trung bình: {d['mean']:.6f}; trung vị: {d['median']:.6f}")
print(f"M0 — nhỏ nhất: {d['min']:.6f}; lớn nhất: {d['max']:.6f}")
print(f"D không lớn hơn bộ gốc: {c['no_higher_D_count']}/{c['n']} bộ hợp lệ")
print('Đối chiếu phụ cùng khoảng chiều dài:', report['secondary_length_matched_comparison'])


### 4. Xem biểu đồ và bộ dữ liệu kết quả

`random_sets.csv` giữ đủ ID và chỉ tiêu từng bộ; `metric_summary.csv` tổng hợp; `route_frequency.csv` ghi tần suất tuyến. Đây là đối chứng M0 tiền khảo sát trên pool cố định, không thay thế số đo thực tế hay M1–M5.


In [ ]:
from IPython.display import Image, display

if 'plot' in report['paths']:
    display(Image(filename=str(input_path(WORK, report['paths']['plot']))))
for name in ('sets', 'summary', 'frequency', 'report', 'protocol', 'inputs'):
    print(name, ':', input_path(WORK, report['paths'][name]))


### 5. Vẽ và phân tích biểu đồ riêng

[Notebook 11 — biểu đồ đối chứng](11_random_comparison_plots.ipynb) chỉ đọc M0 đã lưu để chia khoảng D và xem quan hệ D với độ phủ/chiều dài; không chạy lại bộ điều phối hoặc bốc mẫu. [Hướng dẫn](../docs/lez/RANDOM_COMPARISON_PLOTS.md).